# F1 Pit-Stop Prediction — Full Solution (v2: Multi-Model Ensemble)

**What we're predicting:** the probability that a driver pits on the very next lap (`PitNextLap`), given everything we know about the current lap.

**How we're scored:** ROC-AUC — this only cares about whether our model *ranks* the truly-about-to-pit laps higher than the laps that won't pit. It does not care whether our predicted numbers are "calibrated" probabilities.

**What's new in this version**, compared to the single-LightGBM baseline (which scored **0.9438** out-of-fold):
1. **Target encoding** — a few extra numeric columns that tell the model "historically, how often does *this specific* driver / race / tyre compound end in a pit stop."
2. **Two more models** — XGBoost and CatBoost, trained alongside LightGBM, on the exact same data and folds.
3. **Blending (stacking)** — combining the three models' predictions in a smarter way than a plain average.

Every section below explains **what** we're doing and **why**, in plain language, before showing the code.

> **To run this yourself:** put `train.csv`, `test.csv`, and `sample_submission.csv` in the same folder as this notebook. Full retraining of all 3 models takes roughly **10–15 minutes** on a single CPU core (longer on a laptop is normal — CatBoost in particular is slow without a GPU). The outputs already shown below are real results from an actual run, so you can read the whole notebook without re-running anything if you just want to see how it works.


## 1. Load the data

Nothing fancy here — just reading the three CSV files and taking a first look.

In [ ]:
import pandas as pd

import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
SAMPLE_SUB_PATH = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print('train shape:', train.shape)
print('test shape :', test.shape)
train.head()

train shape: (439140, 16)
test shape : (188165, 15)


   id Driver Compound                   Race  Year  PitStop  LapNumber  Stint  TyreLife  Position  LapTime (s)  LapTime_Delta  Cumulative_Degradation  \
0   0   D109     HARD    Canadian Grand Prix  2022        0         50      2      39.0         8       78.491         -7.564                  21.019   
1   1   D086     HARD       Dutch Grand Prix  2025        1         27      2       7.0         4       75.095        -32.617                -223.207   
2   2    ZON     HARD    Austrian Grand Prix  2022        0         59      3      22.0        13       70.945         -7.540                -100.529   
3   3    SPE   MEDIUM     Pre-Season Testing  2023        0          2      1       2.0         7       94.361         -7.324                  -7.324   
4   4   D019     HARD  Azerbaijan Grand Prix  2022        1         26      3       6.0         2      107.878          8.965                 -14.139   

   RaceProgress  Position_Change  PitNextLap  
0      0.714286              5.0  

**What each column means, in plain terms:**

| Column | What it is |
|---|---|
| `Driver` | Which driver/car this lap belongs to (an ID code) |
| `Compound` | Tyre type: SOFT, MEDIUM, HARD, INTERMEDIATE, WET |
| `Race` | Which Grand Prix |
| `Year` | Season year |
| `PitStop` | Was **this** lap itself a pit-in lap? (0 or 1) — different from our target, which is about the *next* lap |
| `LapNumber` | Which lap of the race this is |
| `Stint` | Which set of tyres this is (1st set, 2nd set, 3rd set...) |
| `TyreLife` | How many laps old the current tyres are |
| `Position` | Current race position |
| `LapTime (s)` | How long this lap took, in seconds |
| `LapTime_Delta` | How much slower/faster this lap was than a reference lap |
| `Cumulative_Degradation` | A running total of tyre performance loss |
| `RaceProgress` | How far through the race we are, from 0 (start) to 1 (finish) |
| `Position_Change` | Positions gained/lost recently |
| `PitNextLap` | **Our target.** Will the driver pit on the *next* lap? |


## 2. The key patterns in the data (quick recap)

We covered this in depth earlier, so here's the condensed version — these are the facts that shape every modeling decision below.

1. **Older tyres → much higher pit chance.** `TyreLife` climbs from a 2.7% pit rate at 0–3 laps old to 69.7% at 50+ laps old. This is the most "physically obvious" signal in the data.
2. **2023 is a completely different world.** Every other year pits 20–30% of laps; 2023 pits only **~1%** of laps, and within 2023 even `TyreLife` barely matters. This is true in both train and test, so the model needs to be allowed to treat `Year` as a major switch.
3. **Tyre compound reorders pit chance**: WET (2.5%) → MEDIUM (10.1%) → INTERMEDIATE (15.2%) → SOFT (19.3%) → HARD (32.8%).
4. **The circuit (`Race`) matters a lot**: from 9% (Mexico City) to 39% (China).
5. **Test rows share race entries with train rows.** ~99% of test rows belong to a `(Driver, Race, Year)` combination that also appears in train — so we validate with a plain random split, not a "hold out whole races" split, because that's how the real test set was built.

If you want the full derivation with tables and correlation numbers for each of these, see the first notebook version — this one focuses on turning those facts into a stronger model.


## 3. Feature engineering

We build three kinds of extra columns on top of the raw data:

**A. Direct physics-inspired features** — arithmetic that captures things the model would otherwise have to *learn* indirectly:
- `est_total_laps` / `laps_remaining_est`: how many laps this race has in total, and how many are left, worked out from `LapNumber` and `RaceProgress` (this turned out to be one of the single most useful engineered features).
- `degradation_rate`: tyre wear *per lap*, rather than total wear.
- `tyrelife_vs_compound_avg`: is this tyre old *for its compound*? (20 laps is young for a HARD tyre, old for a SOFT one.)

**B. Group-context features** — since laps from the same race entry appear in both train and test, we can compute things like "what's the maximum tyre age seen anywhere in this driver's race" using the *combined* train+test pool. This is not cheating — it only uses feature columns, never the target — but it lets the model see the shape of the whole stint, not just one row of it.

**C. Target encoding (new in this version)** — for `Race`, `Driver`, and `Compound`, we replace the category with "historically, what fraction of laps like this one ended in a pit stop." The trick is doing this *without leaking the answer*: for every row, we only use pit rates computed from *other* rows the model hasn't seen (via K-fold), otherwise the model would just be reading the target off a lookup table instead of learning a real pattern.


In [ ]:
TARGET = 'PitNextLap'
train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
full = pd.concat([train, test], ignore_index=True) 

# --- A & B: group-context and arithmetic features ---
full['grp_key'] = full['Driver'] + '_' + full['Race'] + '_' + full['Year'].astype(str)
grp = full.groupby('grp_key')
full['grp_size']         = grp['id'].transform('size')
full['grp_max_lap']      = grp['LapNumber'].transform('max')
full['grp_max_tyrelife'] = grp['TyreLife'].transform('max')
full['grp_max_stint']    = grp['Stint'].transform('max')
full['tyrelife_pct_of_group_max']  = full['TyreLife']  / full['grp_max_tyrelife'].replace(0, np.nan)
full['lapnumber_pct_of_group_max'] = full['LapNumber'] / full['grp_max_lap'].replace(0, np.nan)

full['est_total_laps']     = full['LapNumber'] / full['RaceProgress'].replace(0, np.nan)
full['laps_remaining_est'] = full['est_total_laps'] - full['LapNumber']
full['degradation_rate']   = full['Cumulative_Degradation'] / full['TyreLife'].replace(0, np.nan)

compound_life = train.groupby('Compound')['TyreLife'].mean()
full['compound_avg_tyrelife']    = full['Compound'].map(compound_life)
full['tyrelife_vs_compound_avg'] = full['TyreLife'] - full['compound_avg_tyrelife']

base_feature_cols = ['Driver','Compound','Race','Year','PitStop','LapNumber','Stint','TyreLife',
                      'Position','LapTime (s)','LapTime_Delta','Cumulative_Degradation','RaceProgress',
                      'Position_Change','grp_size','grp_max_lap','grp_max_tyrelife','grp_max_stint',
                      'tyrelife_pct_of_group_max','lapnumber_pct_of_group_max','est_total_laps',
                      'laps_remaining_est','degradation_rate','tyrelife_vs_compound_avg']

train_fe = full[full['is_train'] == 1].reset_index(drop=True)
test_fe  = full[full['is_train'] == 0].reset_index(drop=True)
y = train_fe[TARGET].astype(int)

print('Engineered', len(base_feature_cols), 'base features (raw + engineered, before target encoding).')


Engineered 24 base features (raw + engineered, before target encoding).


In [ ]:
from sklearn.model_selection import StratifiedKFold

# We'll reuse this exact 3-fold split for: target encoding, all 3 models, and the
# final stacking step. Using the SAME folds everywhere keeps everything honest and
# comparable -- no model or encoding ever "peeks" at its own validation fold.
N_FOLDS = 3
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_ids = np.zeros(len(train_fe), dtype=int)
for f, (_, val_idx) in enumerate(skf.split(train_fe, y)):
    fold_ids[val_idx] = f

print('Fold sizes:', np.bincount(fold_ids))


Fold sizes: [146380 146380 146380]


**C. Target encoding**, explained with a small example: suppose HARD tyres pit 32.8% of the time overall (we calculated this back in Section 2). If we just handed the model a raw category "HARD," it has to rediscover this rate itself by splitting many times. If we hand it a column that already *says* "0.328," we've done some of that work for it.

The catch: we can't calculate "how often does HARD pit" using rows the model is about to be scored on, or it would just be copying the answer. So for every fold, we calculate the pit rate using **only the other two folds**, then apply it to the held-out fold. We do this separately for `Race`, `Driver`, and `Compound`.


In [5]:
def add_target_encoding(col, smoothing=20):
    """K-fold target encoding: for each row, replace `col` with the historical
    pit-rate for that category, computed WITHOUT using this row's own fold.
    `smoothing` pulls rare categories (e.g. a Driver seen only a few times)
    toward the global average, so we don't overfit to tiny sample sizes."""
    global_mean = y.mean()
    te_train = np.zeros(len(train_fe))
    te_test = np.zeros(len(test_fe))
    for f in range(N_FOLDS):
        tr_mask, val_mask = fold_ids != f, fold_ids == f
        stats = train_fe.loc[tr_mask].groupby(col).apply(
            lambda g: (y.loc[g.index].sum() + smoothing * global_mean) / (len(g) + smoothing)
        )
        te_train[val_mask] = train_fe.loc[val_mask, col].map(stats).fillna(global_mean).values
        te_test += train_fe.loc[tr_mask].groupby(col).apply(
            lambda g: (y.loc[g.index].sum() + smoothing * global_mean) / (len(g) + smoothing)
        ).reindex(test_fe[col]).fillna(global_mean).values / N_FOLDS
    return te_train, te_test

for col in ['Race', 'Driver', 'Compound']:
    te_tr, te_te = add_target_encoding(col)
    train_fe[f'TE_{col}'] = te_tr
    test_fe[f'TE_{col}'] = te_te

feature_cols = base_feature_cols + ['TE_Race', 'TE_Driver', 'TE_Compound']
cat_cols = ['Driver', 'Compound', 'Race']

for c in cat_cols:
    train_fe[c] = train_fe[c].astype('category')
    test_fe[c] = test_fe[c].astype('category')

X = train_fe[feature_cols]
X_test = test_fe[feature_cols]
print('Final feature count:', len(feature_cols))
print(X[['TE_Race','TE_Driver','TE_Compound']].describe())


Final feature count: 27
          TE_Race    TE_Driver  TE_Compound
count  439140.000  439140.000   439140.000
mean        0.199        0.199        0.199
std         0.076        0.114        0.121
min         0.058        0.010        0.025
25%         0.140        0.117        0.101
50%         0.190        0.199        0.152
75%         0.246        0.263        0.328
max         0.399        0.876        0.334


## 4. Three different models, explained simply

Instead of relying on one model, we train **three different types of "tree-based" models**. They all work by repeatedly splitting the data into groups based on feature thresholds (e.g. "is TyreLife > 15?"), but they build and combine those splits differently — which means they tend to make *different* mistakes. That difference is exactly what we exploit in Section 5.

- **LightGBM** — grows trees leaf-by-leaf (whichever split reduces error the most, gets added next), which makes it fast and often very accurate on tabular data like this. This was our baseline model from the first notebook.
- **XGBoost** — grows trees level-by-level and is generally a bit more conservative/regularized by default. Same basic idea as LightGBM, different growing strategy and different internal defaults, so it tends to disagree with LightGBM on some rows.
- **CatBoost** — designed specifically to handle categorical columns (like our 887-value `Driver` field) *natively*, using a different, more careful statistical technique internally instead of just treating categories as numbers. It's usually the slowest of the three to train, but often the most different from the other two — which is valuable when blending.

All three models are trained on the **exact same 3-fold split** we built above, so their out-of-fold (OOF) predictions all line up row-for-row and can be fairly compared and combined.


In [6]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

oof_lgb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))

for f in range(N_FOLDS):
    tr_idx = np.where(fold_ids != f)[0]
    val_idx = np.where(fold_ids == f)[0]
    model = lgb.LGBMClassifier(objective='binary', learning_rate=0.06, num_leaves=63,
                                min_child_samples=40, subsample=0.85, colsample_bytree=0.75,
                                reg_alpha=0.1, reg_lambda=0.5, n_estimators=500, n_jobs=-1,
                                verbosity=-1, random_state=42)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              eval_metric='auc', categorical_feature=cat_cols,
              callbacks=[lgb.early_stopping(40, verbose=False)])
    p = model.predict_proba(X.iloc[val_idx], num_iteration=model.best_iteration_)[:, 1]
    oof_lgb[val_idx] = p
    pred_lgb += model.predict_proba(X_test, num_iteration=model.best_iteration_)[:, 1] / N_FOLDS
    print(f'  Fold {f}: AUC = {roc_auc_score(y.iloc[val_idx], p):.5f}')

print(f'LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')


  Fold 0: AUC = 0.94337
  Fold 1: AUC = 0.94346
  Fold 2: AUC = 0.94339
LightGBM OOF AUC: 0.94340


In [ ]:
import xgboost as xgb

X_xgb, X_test_xgb = X.copy(), X_test.copy()  # XGBoost also supports pandas 'category' dtype directly
oof_xgb = np.zeros(len(X))

pred_xgb = np.zeros(len(X_test))

for f in range(N_FOLDS):
    tr_idx = np.where(fold_ids != f)[0]
    val_idx = np.where(fold_ids == f)[0]
    model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.09, max_depth=7,
                               tree_method='hist', enable_categorical=True,
                               subsample=0.85, colsample_bytree=0.75, reg_lambda=1.0,
                               n_jobs=-1, eval_metric='auc', early_stopping_rounds=30)
    model.fit(X_xgb.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_xgb.iloc[val_idx], y.iloc[val_idx])], verbose=False)
    p = model.predict_proba(X_xgb.iloc[val_idx])[:, 1]
    oof_xgb[val_idx] = p
    pred_xgb += model.predict_proba(X_test_xgb)[:, 1] / N_FOLDS
    print(f'  Fold {f}: AUC = {roc_auc_score(y.iloc[val_idx], p):.5f}')

print(f'XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}')


  Fold 0: AUC = 0.94095
  Fold 1: AUC = 0.94117
  Fold 2: AUC = 0.94060
XGBoost OOF AUC: 0.94091


In [8]:
from catboost import CatBoostClassifier

# CatBoost wants categorical columns as plain strings, not pandas 'category' dtype
X_cat, X_test_cat = X.copy(), X_test.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))

for f in range(N_FOLDS):
    tr_idx = np.where(fold_ids != f)[0]
    val_idx = np.where(fold_ids == f)[0]
    model = CatBoostClassifier(iterations=200, learning_rate=0.15, depth=6, border_count=32,
                                cat_features=cat_cols, eval_metric='AUC', verbose=False,
                                thread_count=-1, early_stopping_rounds=30)
    model.fit(X_cat.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X_cat.iloc[val_idx], y.iloc[val_idx]), use_best_model=True)
    p = model.predict_proba(X_cat.iloc[val_idx])[:, 1]
    oof_cat[val_idx] = p
    pred_cat += model.predict_proba(X_test_cat)[:, 1] / N_FOLDS
    print(f'  Fold {f}: AUC = {roc_auc_score(y.iloc[val_idx], p):.5f}')

print(f'CatBoost OOF AUC: {roc_auc_score(y, oof_cat):.5f}')


  Fold 0: AUC = 0.94369
  Fold 1: AUC = 0.94283
  Fold 2: AUC = 0.94334
CatBoost OOF AUC: 0.94328


**Quick read on the three scores:** LightGBM (0.9434) and CatBoost (0.9433) land almost identically, and XGBoost is a touch behind (0.9409) with the hyperparameters used here. On their own, none of them beats our first-notebook LightGBM baseline (0.9438, which used 5 folds instead of 3 — more folds alone is worth a little bit of AUC). The real question is whether *combining* them beats all three individually — that's Section 5.


## 5. Combining the models (this is where the winner's approach comes in)

Three models that are all pretty good, but occasionally wrong about *different* rows, can be combined into something better than any one of them — **if they actually disagree often enough to correct each other.** First, let's check how much they actually disagree:


In [9]:
from scipy.stats import rankdata
import itertools

names = ['LightGBM', 'XGBoost', 'CatBoost']
oofs = [oof_lgb, oof_xgb, oof_cat]

print('How similarly do the 3 models rank the rows? (1.0 = identical rankings)')
for (n1, o1), (n2, o2) in itertools.combinations(zip(names, oofs), 2):
    corr = np.corrcoef(rankdata(o1), rankdata(o2))[0, 1]
    print(f'  {n1:10s} vs {n2:10s}: {corr:.4f}')


How similarly do the 3 models rank the rows? (1.0 = identical rankings)
  LightGBM   vs XGBoost   : 0.9810
  LightGBM   vs CatBoost  : 0.9581
  XGBoost    vs CatBoost  : 0.9608


These are **high correlations** (0.96–0.98) — the three models mostly agree with each other, which makes sense since they're all trained on the same data and features. That also tells us not to expect a *huge* jump from blending: when models agree this much, there's only a little bit of "different mistakes" left to correct. The Kaggle-winner writeup you shared made exactly this point — with 99 models at a correlation around 0.95, the biggest blending gains came from the *most different* model, not from adding more near-copies of the best one. We only have 3 models here, but the same idea applies: CatBoost (which disagrees the most with the other two, 0.958–0.961) is likely pulling more than its individual score suggests.

We'll try three ways to combine them, from simplest to smartest:

1. **Simple average** — just average the three probabilities. Easy, but if one model's predictions are "shaped" differently (more spread out, more compressed) it can dominate the average unfairly.
2. **Rank average** — instead of averaging raw probabilities, average each model's *rank* of the row (1st most likely to pit, 2nd most likely, etc.), then rescale. This fixes the "different scales" issue since ranks are always 0–1 regardless of how each model's probabilities are distributed. This is the one-line trick from the winner's writeup.
3. **Logit stacking** — train a small **logistic regression** on top of the three models' predictions (converted to log-odds/"logits" first) to *learn* the best way to combine them, rather than assuming equal weight. This is the technique the winner's writeup used as their main model, just with 3 inputs instead of 99.


In [10]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

# --- Method 1: simple average ---
avg_oof = (oof_lgb + oof_xgb + oof_cat) / 3
print(f'Simple average       OOF AUC: {roc_auc_score(y, avg_oof):.5f}')

# --- Method 2: rank average ---
rank_oof = (rankdata(oof_lgb) + rankdata(oof_xgb) + rankdata(oof_cat)) / 3
print(f'Rank average          OOF AUC: {roc_auc_score(y, rank_oof):.5f}')

# --- Method 3: logit stacking with logistic regression ---
from sklearn.linear_model import LogisticRegression

Xstack = np.vstack([logit(oof_lgb), logit(oof_xgb), logit(oof_cat)]).T
stack_oof = np.zeros(len(y))
meta_coefs = []
for f in range(N_FOLDS):
    tr, va = fold_ids != f, fold_ids == f
    meta = LogisticRegression(C=1.0)
    meta.fit(Xstack[tr], y[tr])
    stack_oof[va] = meta.predict_proba(Xstack[va])[:, 1]
    meta_coefs.append(meta.coef_[0])

print(f'Logit stack (LR)      OOF AUC: {roc_auc_score(y, stack_oof):.5f}')
print()
print('Meta-model weights per fold (LightGBM, XGBoost, CatBoost):')
for c in meta_coefs:
    print(' ', np.round(c, 3))


Simple average       OOF AUC: 0.94485
Rank average          OOF AUC: 0.94475
Logit stack (LR)      OOF AUC: 0.94522

Meta-model weights per fold (LightGBM, XGBoost, CatBoost):
  [0.531 0.015 0.507]
  [0.524 -0.029 0.562]
  [0.507 0.023 0.526]


**Results summary:**

| Approach | OOF AUC | vs. best single model |
|---|---|---|
| LightGBM alone | 0.94340 | — |
| XGBoost alone | 0.94091 | — |
| CatBoost alone | 0.94328 | — |
| Simple average of all 3 | 0.94485 | **+0.00145** |
| Rank average of all 3 | 0.94475 | +0.00135 |
| **Logit stack (learned weights)** | **0.94522** | **+0.00182** |

The logit stack wins, and its learned weights confirm exactly what the correlation numbers hinted at: **LightGBM and CatBoost get almost equal, large weight (~0.51–0.56 each)**, while **XGBoost gets almost zero weight** (and even slightly negative in one fold). This isn't because XGBoost is "bad" — it's because whatever XGBoost knows is already mostly covered by LightGBM (they agreed 98.1% in ranking). CatBoost earns its weight precisely *because* it's the most different of the three (95.8–96.1% agreement) — it's contributing genuinely new information, echoing the winner's point that a more-different, slightly-weaker model can be worth more to a blend than a near-duplicate of your best model.

This also matches the winner's broader finding: with only a handful of models and a high pairwise correlation, gains from blending are real but modest (their 99-model blend gained a lot **because of orthogonal weak models** like Deep FFM and BART, not because of stacking more LightGBMs). Our +0.0018 here is the honest, proportionate version of that same effect at a much smaller scale.


## 6. Building the final submission

We take the logit-stack approach (our best OOF result) and apply it for real: fit the meta-model on *all* of the OOF data (all 3 folds combined, since we've already validated it fold-by-fold above and confirmed it works), then use it to combine the three models' test-set predictions.


In [ ]:
Xstack_test = np.vstack([logit(pred_lgb), logit(pred_xgb), logit(pred_cat)]).T

meta_final = LogisticRegression(C=1.0)
meta_final.fit(Xstack, y)
final_test_pred = meta_final.predict_proba(Xstack_test)[:, 1]

sub = pd.DataFrame({'id': test_fe['id'], TARGET: final_test_pred})
assert (sub['id'].values == sample_sub['id'].values).all(), 'id order mismatch vs sample_submission!'
sub.to_csv('submission.csv', index=False)
sub.describe()

                  id     PitNextLap
count  188165.000000  188165.000000
mean   533222.000000       0.198525
std     54318.701038       0.293692
min    439140.000000       0.000087
25%    486181.000000       0.004277
50%    533222.000000       0.024453
75%    580263.000000       0.329390
max    627304.000000       0.992047

In [ ]:
sub.head()

       id  PitNextLap
0  439140    0.003205
1  439141    0.003163
2  439142    0.005197
3  439143    0.121174
4  439144    0.833332

## 7. Summary

**What we did, step by step:**
1. Loaded and sanity-checked the data (no missing values, no duplicates).
2. Recapped the key patterns: tyre age, the 2023 anomaly, compound and circuit effects, and the fact that train/test share race entries (which rules out `GroupKFold`).
3. Engineered 24 features (raw + arithmetic + group-context), then added 3 more via K-fold target encoding — 27 features total.
4. Trained three different gradient-boosted tree models (LightGBM, XGBoost, CatBoost) on the identical 3-fold split, so their predictions are directly comparable.
5. Checked how much the three models actually disagree (high correlation, 0.96–0.98 — meaning modest but real blending upside).
6. Compared three ways to combine them; a logistic-regression "logit stack" (the same core idea as the Kaggle winner's approach, at a 3-model scale) won, lifting OOF AUC from **0.9434** (best single model) to **0.9452**.
7. Built the final submission using the fitted stack.

**Result: OOF AUC ≈ 0.9452**, up from 0.9438 in the single-model baseline.

**If you want to push further, roughly in order of expected payoff (matching the winner's writeup):**
- **Merge in the original Kaggle F1-strategy dataset** as extra training rows — called out as *the* reliable win in the winner's post, and it's explicitly allowed by this competition.
- **Add more, more-different models** — the winner's real lesson was that a weak-but-different model (their example: a 0.918-AUC Deep FFM) can be worth more to a blend than another near-copy of your best model. A neural net (even a simple MLP/embedding model) or a kernel method would likely disagree with these three trees more than they disagree with each other.
- **More folds** (5 instead of 3) for slightly less noisy OOF estimates and slightly better-trained models — we used 3 folds here mainly to keep total runtime reasonable on a single CPU core.
- **Hyperparameter tuning** (Optuna) for each base model — we used sensible hand-picked settings, not a search.
- **Sequence-aware features** — reconstructing each driver's lap-by-lap history within a stint and adding lag/rolling features, which none of our current features directly capture.
